In [ ]:
import pandas as pd
import numpy as np

In [ ]:
dataset_a = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_A.csv')
dataset_b = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_B.csv')
dataset_c = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_C.csv')
dataset_x = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_X.csv')

In [ ]:
dataset = pd.concat([dataset_a, dataset_b, dataset_c, dataset_x])

In [ ]:
dataset.head()

,MeanRR,SD_RR,RMSSD,NN50,pNN50,AverageHeartRate,StandardDeviationHeartRate,mean_R_Peak_Amplitudes,QRS_Duration,QRS_Amplitude,QRS_Slope,LF_HF_Ratio,PSE,Label,Record_ID
0,0.900000,0.059666,0.041805,15,22.727273,66.957178,4.464194,0.007658,0.093134,0.006705,0.073050,6.885354,7.846725,N,a01
1,0.838714,0.046311,0.037879,11,15.714286,71.760662,4.107388,0.007787,0.094225,0.006580,0.071032,2.563174,7.774343,N,a01
2,0.810139,0.080535,0.042410,13,18.055556,74.787540,7.436019,0.007647,0.092055,0.006349,0.070468,0.402439,7.888780,N,a01
3,0.748205,0.061470,0.027868,8,10.256410,80.765865,7.110214,0.007194,0.090886,0.006453,0.071624,0.460587,7.867083,N,a01
4,0.794595,0.047085,0.031579,8,10.810811,75.770464,4.469915,0.007040,0.093200,0.006326,0.068940,1.423685,7.742370,N,a01


In [ ]:
dataset = dataset.dropna()

In [ ]:
dataset.shape

(34198, 15)

In [ ]:
# from sklearn.model_selection import train_test_split

# unique_patients = dataset['Record_ID'].unique()

# # Split patients into Train (80%) and Test (20%)
# train_patients, test_patients = train_test_split(unique_patients, test_size=0.3, random_state=42)

# # Create train & test datasets
# train_data = dataset[dataset['Record_ID'].isin(train_patients)]
# test_data = dataset[dataset['Record_ID'].isin(test_patients)]

# # Save or return the datasets
# train_data.to_csv("train_patient_wise.csv", index=False)
# test_data.to_csv("test_patient_wise.csv", index=False)

# print("Train Data Patients:", len(train_patients))
# print("Test Data Patients:", len(test_patients))
# print("Train Size:", train_data.shape, "Test Size:", test_data.shape)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split


# Extract features and labels

X = dataset.drop(columns=["Label", "Record_ID"])
y = dataset["Label"]

# X_train , X_test , y_train , y_test = train_data.drop(columns=["Label", "Record_ID"]), test_data.drop(columns=["Label", "Record_ID"]), train_data["Label"], test_data["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
# Apply ADASYN to balance the dataset
adasyn = ADASYN(random_state=75)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)
# X_test, y_test = adasyn.fit_resample(X_test, y_test)



In [ ]:
print(dataset['Label'].tolist().count('A'))
print(dataset['Label'].tolist().count('N'))

13059
21139


In [ ]:
dataset.head()

,MeanRR,SD_RR,RMSSD,NN50,pNN50,AverageHeartRate,StandardDeviationHeartRate,mean_R_Peak_Amplitudes,QRS_Duration,QRS_Amplitude,QRS_Slope,LF_HF_Ratio,PSE,Label,Record_ID
0,0.900000,0.059666,0.041805,15,22.727273,66.957178,4.464194,0.007658,0.093134,0.006705,0.073050,6.885354,7.846725,N,a01
1,0.838714,0.046311,0.037879,11,15.714286,71.760662,4.107388,0.007787,0.094225,0.006580,0.071032,2.563174,7.774343,N,a01
2,0.810139,0.080535,0.042410,13,18.055556,74.787540,7.436019,0.007647,0.092055,0.006349,0.070468,0.402439,7.888780,N,a01
3,0.748205,0.061470,0.027868,8,10.256410,80.765865,7.110214,0.007194,0.090886,0.006453,0.071624,0.460587,7.867083,N,a01
4,0.794595,0.047085,0.031579,8,10.810811,75.770464,4.469915,0.007040,0.093200,0.006326,0.068940,1.423685,7.742370,N,a01


In [ ]:
X_train_resampled.shape

(25693, 13)

In [ ]:
y_train_resampled.shape

(25693,)

In [ ]:
print(y_train_resampled.tolist().count('A'))
print(y_train_resampled.tolist().count('N'))

13034
12659


In [ ]:
e = 10

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, SimpleRNN,LSTM


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# Label Encoding
labelEncoder = LabelEncoder()
y_train_encoded = labelEncoder.fit_transform(y_train_resampled)
y_test_encoded = labelEncoder.transform(y_test)

# Reshape for CNN/RNN/LSTM models
X_train_final = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_final = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

# Split test data further into validation and test
from sklearn.model_selection import train_test_split

# Separate Apnea (A) and Normal (N) samples
X_apnea = X_test_final[y_test_encoded == 1]  # Apnea class
X_normal = X_test_final[y_test_encoded == 0]  # Normal class

y_apnea = y_test_encoded[y_test_encoded == 1]
y_normal = y_test_encoded[y_test_encoded == 0]

# Split each class into 50% Validation and 50% Final Test
X_apnea_val, X_apnea_test, y_apnea_val, y_apnea_test = train_test_split(X_apnea, y_apnea, test_size=0.5, random_state=42)
X_normal_val, X_normal_test, y_normal_val, y_normal_test = train_test_split(X_normal, y_normal, test_size=0.5, random_state=42)

# Merge back to form balanced validation and test sets
X_val = np.concatenate([X_apnea_val, X_normal_val]) # Change pd.concat to np.concatenate
X_test_final = np.concatenate([X_apnea_test, X_normal_test]) # Change pd.concat to np.concatenate

y_val = np.concatenate([y_apnea_val, y_normal_val]) # Change pd.concat to np.concatenate
y_test_final = np.concatenate([y_apnea_test, y_normal_test]) # Change pd.concat to np.concatenate

# Shuffle data to mix classes well

from sklearn.utils import shuffle
X_val, y_val = shuffle(X_val, y_val, random_state=42)

# Similarly, use shuffle for X_test_final and y_test_final
X_test_final, y_test_final = shuffle(X_test_final, y_test_final, random_state=42)
# Print confirmation
print("Validation Set - Apnea:", sum(y_val == 1), "Normal:", sum(y_val == 0))
print("Final Test Set - Apnea:", sum(y_test_final == 1), "Normal:", sum(y_test_final == 0))


print(f"Training Data Shape: {X_train_final.shape}, {y_train_encoded.shape}")
print(f"Validation Data Shape: {X_val.shape}, {y_val.shape}")
print(f"Test Data Shape: {X_test_final.shape}, {y_test_final.shape}")

Validation Set - Apnea: 4240 Normal: 2600
Final Test Set - Apnea: 4240 Normal: 2600
Training Data Shape: (25693, 13, 1), (25693,)
Validation Data Shape: (6840, 13, 1), (6840,)
Test Data Shape: (6840, 13, 1), (6840,)


In [ ]:
cnn_model = Sequential([
    Conv1D(filters=128, kernel_size=3, activation='relu', input_shape=(X_train_final.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_model.fit(X_train_final, y_train_encoded, epochs = 10, batch_size=32, validation_data=(X_val, y_val))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.6819 - loss: 0.5938 - val_accuracy: 0.7842 - val_loss: 0.4530
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7597 - loss: 0.4963 - val_accuracy: 0.7674 - val_loss: 0.4622
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7765 - loss: 0.4697 - val_accuracy: 0.7765 - val_loss: 0.4517
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7883 - loss: 0.4559 - val_accuracy: 0.7962 - val_loss: 0.4154
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7947 - loss: 0.4386 - val_accuracy: 0.8099 - val_loss: 0.3932
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7931 - loss: 0.4423 - val_accuracy: 0.8072 - val_loss: 0.3989
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7982 - loss: 0.4291 - val_accuracy: 0.8058 - val_loss: 0.3912
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8066 - loss: 0.4213 - val_accuracy: 0

In [ ]:
rnn_model = Sequential([
    SimpleRNN(128, activation='relu', return_sequences=True, input_shape=(X_train_final.shape[1], 1)),
    Dropout(0.3),
    SimpleRNN(64, activation='relu'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


803/803 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.7346 - loss: 0.5373 - val_accuracy: 0.7860 - val_loss: 0.4473
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7946 - loss: 0.4440 - val_accuracy: 0.7981 - val_loss: 0.4182
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8111 - loss: 0.4116 - val_accuracy: 0.8276 - val_loss: 0.3784
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8204 - loss: 0.3946 - val_accuracy: 0.8145 - val_loss: 0.3799
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8253 - loss: 0.3879 - val_accuracy: 0.8113 - val_loss: 0.4071
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8298 - loss: 0.3777 - val_accuracy: 0.8294 - val_loss: 0.3649
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8359 - loss: 0.3615 - val_accuracy: 0.8219 - val_loss: 0.3830
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8378 - loss: 0.3583 - val_accuracy: 0.8327 - va

In [ ]:
lstm_model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True, input_shape=(X_train_final.shape[1], 1)),
    Dropout(0.3),
    LSTM(64, activation='tanh'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - accuracy: 0.6173 - loss: 0.6407 - val_accuracy: 0.6627 - val_loss: 0.5775
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7290 - loss: 0.5387 - val_accuracy: 0.7858 - val_loss: 0.4544
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7829 - loss: 0.4613 - val_accuracy: 0.7949 - val_loss: 0.4357
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.7930 - loss: 0.4492 - val_accuracy: 0.8000 - val_loss: 0.4168
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8065 - loss: 0.4222 - val_accuracy: 0.8091 - val_loss: 0.4055
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8179 - loss: 0.4056 - val_accuracy: 0.8108 - val_loss: 0.3983
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8186 - loss: 0.3972 - val_accuracy: 0.8199 - val_loss: 0.3778
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8249 - loss: 0.3851 - val_accura

In [ ]:
cnn_lstm_model = Sequential([
    Conv1D(128, kernel_size=3, activation='relu', input_shape=(X_train_final.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    LSTM(128, activation='tanh'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_lstm_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.6427 - loss: 0.6317 - val_accuracy: 0.7421 - val_loss: 0.4981
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.7373 - loss: 0.5286 - val_accuracy: 0.7702 - val_loss: 0.4537
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7701 - loss: 0.4782 - val_accuracy: 0.7738 - val_loss: 0.4527
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.7843 - loss: 0.4550 - val_accuracy: 0.7927 - val_loss: 0.4259
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8014 - loss: 0.4307 - val_accuracy: 0.8004 - val_loss: 0.4079
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8051 - loss: 0.4244 - val_accuracy: 0.8091 - val_loss: 0.3853
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.8130 - loss: 0.4076 - val_accuracy: 0.8095 - val_loss: 0.3851
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.8126 - loss: 0.4059 - val_accuracy:

In [ ]:
cnn_rnn_model = Sequential([
    Conv1D(128, kernel_size=3, activation='relu', input_shape=(X_train_final.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    SimpleRNN(128, activation='relu'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn_rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn_rnn_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.6740 - loss: 0.5949 - val_accuracy: 0.7738 - val_loss: 0.4609
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7701 - loss: 0.4795 - val_accuracy: 0.8102 - val_loss: 0.4128
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.7821 - loss: 0.4553 - val_accuracy: 0.8145 - val_loss: 0.3929
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7968 - loss: 0.4331 - val_accuracy: 0.7988 - val_loss: 0.4184
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8061 - loss: 0.4187 - val_accuracy: 0.8183 - val_loss: 0.3840
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8098 - loss: 0.4146 - val_accuracy: 0.8113 - val_loss: 0.3994
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8192 - loss: 0.4004 - val_accuracy: 0.8196 - val_loss: 0.3860
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8148 - loss: 0.3960 - val_accuracy: 0

In [ ]:
lstm_rnn_model = Sequential([
    LSTM(128, return_sequences=True, activation='tanh', input_shape=(X_train_final.shape[1], 1)),
    Dropout(0.3),
    SimpleRNN(64, activation='relu'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_rnn_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 25s 28ms/step - accuracy: 0.6819 - loss: 0.5841 - val_accuracy: 0.7750 - val_loss: 0.4688
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.7865 - loss: 0.4585 - val_accuracy: 0.8088 - val_loss: 0.4147
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 42s 26ms/step - accuracy: 0.7987 - loss: 0.4364 - val_accuracy: 0.8094 - val_loss: 0.4054
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 42s 28ms/step - accuracy: 0.8035 - loss: 0.4227 - val_accuracy: 0.8098 - val_loss: 0.3999
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 39s 25ms/step - accuracy: 0.8087 - loss: 0.4152 - val_accuracy: 0.8168 - val_loss: 0.3867
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 27s 33ms/step - accuracy: 0.8212 - loss: 0.4003 - val_accuracy: 0.8225 - val_loss: 0.3726
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 21s 26ms/step - accuracy: 0.8257 - loss: 0.3895 - val_accuracy: 0.8235 - val_loss: 0.3806
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 41s 26ms/step - accuracy: 0.8228 - loss: 0.3869 - 

In [ ]:
lstm_cnn_model = Sequential([
    LSTM(128, return_sequences=True, activation='tanh', input_shape=(X_train_final.shape[1], 1)),
    Dropout(0.3),
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_cnn_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.6671 - loss: 0.6075 - val_accuracy: 0.7674 - val_loss: 0.4707
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.7575 - loss: 0.4994 - val_accuracy: 0.7655 - val_loss: 0.4805
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7759 - loss: 0.4689 - val_accuracy: 0.7984 - val_loss: 0.4122
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.7837 - loss: 0.4560 - val_accuracy: 0.7849 - val_loss: 0.4343
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7910 - loss: 0.4376 - val_accuracy: 0.8047 - val_loss: 0.3996
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.7997 - loss: 0.4282 - val_accuracy: 0.8037 - val_loss: 0.4002
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8017 - loss: 0.4221 - val_accuracy: 0.8177 - val_loss: 0.3788
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.8121 - loss: 0.4076 - val_accu

In [ ]:
rnn_cnn_model = Sequential([
    SimpleRNN(128, return_sequences=True, activation='relu', input_shape=(X_train_final.shape[1], 1)),
    Dropout(0.3),
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

rnn_cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_cnn_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.6927 - loss: 0.5744 - val_accuracy: 0.7542 - val_loss: 0.5075
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7899 - loss: 0.4577 - val_accuracy: 0.7925 - val_loss: 0.4265
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.7995 - loss: 0.4339 - val_accuracy: 0.8146 - val_loss: 0.3897
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8099 - loss: 0.4120 - val_accuracy: 0.8091 - val_loss: 0.4126
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8164 - loss: 0.4027 - val_accuracy: 0.8145 - val_loss: 0.3928
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.8225 - loss: 0.3961 - val_accuracy: 0.8215 - val_loss: 0.3788
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.8319 - loss: 0.3763 - val_accuracy: 0.8164 - val_loss: 0.3728
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8276 - loss: 0.3747 - val_accuracy: 0

In [ ]:
rnn_lstm_model = Sequential([
    SimpleRNN(128, return_sequences=True, activation='relu', input_shape=(X_train_final.shape[1], 1)),
    Dropout(0.3),
    LSTM(64, activation='tanh'),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

rnn_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_lstm_model.fit(X_train_final, y_train_encoded, epochs = e, batch_size=32, validation_data=(X_val, y_val))


Epoch 1/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 23s 25ms/step - accuracy: 0.7140 - loss: 0.5531 - val_accuracy: 0.7933 - val_loss: 0.4275
Epoch 2/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.7990 - loss: 0.4418 - val_accuracy: 0.8215 - val_loss: 0.3879
Epoch 3/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.8074 - loss: 0.4160 - val_accuracy: 0.8222 - val_loss: 0.3838
Epoch 4/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.8163 - loss: 0.4020 - val_accuracy: 0.8357 - val_loss: 0.3628
Epoch 5/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - accuracy: 0.8252 - loss: 0.3888 - val_accuracy: 0.8332 - val_loss: 0.3676
Epoch 6/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 21s 23ms/step - accuracy: 0.8352 - loss: 0.3718 - val_accuracy: 0.8377 - val_loss: 0.3555
Epoch 7/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.8383 - loss: 0.3614 - val_accuracy: 0.8414 - val_loss: 0.3499
Epoch 8/10
803/803 ━━━━━━━━━━━━━━━━━━━━ 23s 25ms/step - accuracy: 0.8389 - loss: 0.3603 - 

In [ ]:
!pip install ace_tools

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

# Function to calculate all performance metrics
def metricsCalculator(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'AUC': roc_auc_score(y_true, y_pred)
    }

# Dictionary to store models
models = {
    'CNN': cnn_model, 'RNN': rnn_model, 'LSTM': lstm_model,
    'CNN-LSTM': cnn_lstm_model, 'CNN-RNN': cnn_rnn_model,
    'LSTM-RNN': lstm_rnn_model, 'LSTM-CNN': lstm_cnn_model,
    'RNN-CNN': rnn_cnn_model, 'RNN-LSTM': rnn_lstm_model
}

metrics_results = []

for name, model in models.items():
    # Evaluate using model.evaluate() to get loss & accuracy
    loss, accuracy = model.evaluate(X_test_final, y_test_final, verbose=0)

    # Get predictions and convert to binary labels (0 or 1)
    y_pred_prob = model.predict(X_test_final)
    y_pred_class = (y_pred_prob > 0.5).astype("int32")

    # Calculate all metrics
    metric_values = metricsCalculator(y_test_final, y_pred_class)

    # Store results in a list
    metrics_results.append({
        'Model': name,
        'Loss': loss,
        'Accuracy': accuracy,
        'Recall': metric_values['Recall'],
        'Precision': metric_values['Precision'],
        'F1-Score': metric_values['F1-Score'],
        'AUC': metric_values['AUC']
    })

# Convert to DataFrame for display
import pandas as pd
metrics_df = pd.DataFrame(metrics_results)


metrics_df


214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step


,Model,Loss,Accuracy,Recall,Precision,F1-Score,AUC
0,CNN,0.398140,0.804240,0.746698,0.922763,0.825446,0.822388
1,RNN,0.345537,0.838158,0.816038,0.913652,0.862090,0.845134
2,LSTM,0.367242,0.825146,0.790330,0.916074,0.848569,0.836127
3,CNN-LSTM,0.369439,0.824269,0.797642,0.907676,0.849109,0.832667
4,CNN-RNN,0.377376,0.819152,0.773821,0.921888,0.841390,0.833449
5,LSTM-RNN,0.362086,0.831871,0.801415,0.916892,0.855273,0.841477
6,LSTM-CNN,0.387556,0.805848,0.751887,0.920323,0.827622,0.822866
7,RNN-CNN,0.367676,0.830263,0.788443,0.926809,0.852045,0.843452
8,RNN-LSTM,0.355177,0.834795,0.803066,0.920270,0.857683,0.844802


In [ ]:
rnn_model.save('rnn_model.h5')
lstm_model.save('lstm_model.h5')
rnn_cnn_model.save('rnn_cnn_model.h5')
rnn_lstm_model.save('rnn_lstm_model.h5')